In [0]:
# 1. Environment & Storage Setup
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.01_bronze")
base_volume = "/Volumes/training_medallion/00_landing/raw_landing_zone"

# Create format-specific landing directories
dbutils.fs.mkdirs(f"{base_volume}/json_landing")
dbutils.fs.mkdirs(f"{base_volume}/csv_landing")
dbutils.fs.mkdirs(f"{base_volume}/log_landing")
dbutils.fs.mkdirs(f"{base_volume}/text_landing")
print("✅ Landing directories successfully created in Unity Catalog Volume!")

# 2. Process 1: Ingesting JSON Data (Customer Support Tickets)
json_payload = """{"ticket_id": "TCK-101", "customer_id": 1, "issue": "Payment failed", "status": "OPEN"}
{"ticket_id": "TCK-102", "customer_id": 2, "issue": "Account locked", "status": "CLOSED"}"""

json_path = f"{base_volume}/json_landing/tickets_batch1.json"
dbutils.fs.put(json_path, json_payload, overwrite=True)
print(f"✅ JSON file saved to: {json_path}")

checkpoint_json = f"{base_volume}/_checkpoints/json_tickets"
schema_json = f"{base_volume}/_schemas/json_tickets"

df_json_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_json)
    .load(f"{base_volume}/json_landing")
)

q_json = (
    df_json_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_json)
    .table("training_medallion.01_bronze.bronze_json_tickets")
)
q_json.awaitTermination()

# 3. Process 2: Ingesting CSV Data (Customer Profiles)
csv_payload = """customer_id,customer_name,region,total_orders\n1,Jenna,APAC,2\n2,John,EMEA,6\n3,Lea,AMER,10"""
csv_path = f"{base_volume}/csv_landing/customers_batch1.csv"
dbutils.fs.put(csv_path, csv_payload, overwrite=True)
print(f"✅ CSV file saved to: {csv_path}")

checkpoint_csv = f"{base_volume}/_checkpoints/csv_customers"
schema_csv = f"{base_volume}/_schemas/csv_customers"

df_csv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_csv)
    .load(f"{base_volume}/csv_landing")
)

q_csv = (
    df_csv_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_csv)
    .table("training_medallion.01_bronze.bronze_csv_customers")
)
q_csv.awaitTermination()

# 4. Ingesting Server Logs (Semi-Structured Application Logs)
log_payload = """2026-10-17 10:15:30 INFO User 1 logged in successfully
2026-10-17 10:16:05 ERROR Payment gateway timeout for ticket TCK-101
2026-10-17 10:17:12 WARN High memory usage detected on node 3"""

log_path = f"{base_volume}/log_landing/app_server.log"
dbutils.fs.put(log_path, log_payload, overwrite=True)
print(f"✅ Log file saved to: {log_path}")

checkpoint_log = f"{base_volume}/_checkpoints/server_logs"

df_log_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .load(f"{base_volume}/log_landing")
)

q_log = (
    df_log_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_log)
    .table("training_medallion.01_bronze.bronze_server_logs")
)
q_log.awaitTermination()

# 5. Ingesting Plain Text Files (Support Transcripts)
text_payload = """Customer Jenna reported an issue regarding order payment.
Agent Rudy transferred the call to technical support.
Resolution provided and ticket closed."""

text_path = f"{base_volume}/text_landing/chat_transcript_101.txt"
dbutils.fs.put(text_path, text_payload, overwrite=True)
print(f"✅ Text file saved to: {text_path}")

checkpoint_text = f"{base_volume}/_checkpoints/chat_text"

df_text_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .load(f"{base_volume}/text_landing")
)

q_text = (
    df_text_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_text)
    .table("training_medallion.01_bronze.bronze_chat_transcripts")
)
q_text.awaitTermination()

✅ Landing directories successfully created in Unity Catalog Volume!
Wrote 177 bytes.
✅ JSON file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/json_landing/tickets_batch1.json
Wrote 88 bytes.
✅ CSV file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/csv_landing/customers_batch1.csv
Wrote 185 bytes.
✅ Log file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/log_landing/app_server.log
Wrote 150 bytes.
✅ Text file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/text_landing/chat_transcript_101.txt
